In [1]:
# ================= MIXED-PRECISION QAT | CIFAR-10 | RESNET50 =================
!pip install torch torchvision timm tqdm --quiet

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import timm
import torch.ao.quantization as quant

# ---------------- Setup ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 100
WARMUP_EPOCHS = 30
BATCH_SIZE = 128

# ---------------- CIFAR-10 ----------------
MEAN = [0.4914, 0.4822, 0.4465]
STD  = [0.2470, 0.2435, 0.2616]

train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=train_tf)
test_ds  = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)

train_ld = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=2)
test_ld  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=2)

# ---------------- Mixed-Precision ResNet50 ----------------
class MixedQATResNet50(nn.Module):
    def __init__(self):
        super().__init__()
        base = timm.create_model("resnet50", pretrained=True, num_classes=10)

        # CIFAR-10 friendly stem
        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()
        nn.init.kaiming_normal_(base.conv1.weight, mode="fan_out", nonlinearity="relu")

        self.conv1 = base.conv1
        self.bn1 = base.bn1
        self.act1 = base.act1

        self.quant = quant.QuantStub()
        self.dequant = quant.DeQuantStub()

        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4

        self.pool = base.global_pool
        self.fc = base.fc

    def forward(self, x):
        # FP32 stem
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.act1(x)

        # Quantized body
        x = self.quant(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.dequant(x)

        # FP32 head
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

model = MixedQATResNet50().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=[60, 85], gamma=0.1
)

# ---------------- Evaluation ----------------
def evaluate(m):
    m.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in test_ld:
            x, y = x.to(device), y.to(device)
            out = m(x)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / total

# ---------------- Training ----------------
best_acc = 0.0
qat_enabled = False

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    loop = tqdm(train_ld, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for x, y in loop:
        x, y = x.to(device), y.to(device)

        out = model(x)
        loss = criterion(out, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        loop.set_postfix(loss=running_loss / (loop.n + 1))

    scheduler.step()
    avg_loss = running_loss / len(train_ld)
    acc = evaluate(model)

    print(f"[Epoch {epoch+1}] Train Loss: {avg_loss:.4f} | Val Acc: {acc:.4f}")

    # ----- Enable QAT -----
    if epoch + 1 == WARMUP_EPOCHS and not qat_enabled:
        print(">>> Enabling Mixed-Precision QAT")
        model.train()
        model.qconfig = quant.get_default_qat_qconfig("fbgemm")
        quant.prepare_qat(model, inplace=True)

        for g in optimizer.param_groups:
            g["lr"] = 1e-4

        qat_enabled = True

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "best_mixed_qat_resnet50_cifar10.pth")
        print(">>> Best model saved")

# ---------------- Convert to REAL INT8 ----------------
model.cpu().eval()
int8_model = quant.convert(model, inplace=False)
torch.save(int8_model.state_dict(), "mixed_qat_resnet50_cifar10_int8.pth")

print("INT8 Mixed-Precision QAT model saved")

100%|██████████| 170M/170M [00:01<00:00, 94.8MB/s] 


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Epoch 1/100: 100%|██████████| 391/391 [02:30<00:00,  2.60it/s, loss=1.68]


[Epoch 1] Train Loss: 1.6755 | Val Acc: 0.5398
>>> Best model saved


Epoch 2/100: 100%|██████████| 391/391 [02:32<00:00,  2.57it/s, loss=0.88] 


...

[Epoch 99] Train Loss: 0.2211 | Val Acc: 0.8998


Epoch 100/100: 100%|██████████| 391/391 [03:05<00:00,  2.10it/s, loss=0.224]


[Epoch 100] Train Loss: 0.2236 | Val Acc: 0.8966


/tmp/ipykernel_55/2471097608.py:162: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  int8_model = quant.convert(model, inplace=False)


INT8 Mixed-Precision QAT model saved


In [7]:
!pwd
!ls -lh

/kaggle/working
total 205M
-rw-r--r-- 1 root root  91M Feb 26 04:41 best_mixed_qat_resnet50_cifar10.pth
drwxr-xr-x 3 root root 4.0K Feb 26 03:26 data
-rw-r--r-- 1 root root  24M Feb 26 08:39 mixed_qat_resnet50_cifar10_int8.pth
-rw-r--r-- 1 root root  91M Feb 26 08:48 model.pt


In [8]:

# Verify saved models in Kaggle working directory
!ls -lh /kaggle/working

total 205M
-rw-r--r-- 1 root root  91M Feb 26 04:41 best_mixed_qat_resnet50_cifar10.pth
drwxr-xr-x 3 root root 4.0K Feb 26 03:26 data
-rw-r--r-- 1 root root  24M Feb 26 08:39 mixed_qat_resnet50_cifar10_int8.pth
-rw-r--r-- 1 root root  91M Feb 26 08:48 model.pt
